## Task 1:

In [1]:
# Total cards in deck
total_cards = 52

#1. Probability of drawing a red card (hearts or diamonds)
red_cards = 26  # 13 hearts + 13 diamonds
p_red = red_cards / total_cards

#2. Probability of heart given that the card is red
hearts = 13
p_heart_given_red = hearts / red_cards  # Out of 26 red cards

#3. Probability that a face card is a diamond
face_cards = 12 
diamond_faces = 3  
p_diamond_given_face = diamond_faces / face_cards

#4. Probability that a face card is a spade or a queen
spade_faces = 3 
queens = 4       
queen_of_spades = 1  # Overlap
spade_or_queen = spade_faces + queens - queen_of_spades  # Inclusion-exclusion
p_spade_or_queen_given_face = spade_or_queen / face_cards

print(f"1. P(Red card) = {p_red:.2f}")
print(f"2. P(Heart | Red) = {p_heart_given_red:.2f}")
print(f"3. P(Diamond | Face card) = {p_diamond_given_face:.2f}")
print(f"4. P(Spade or Queen | Face card) = {p_spade_or_queen_given_face:.2f}")

1. P(Red card) = 0.50
2. P(Heart | Red) = 0.50
3. P(Diamond | Face card) = 0.25
4. P(Spade or Queen | Face card) = 0.50


## Task 2:

In [6]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

In [5]:
# Step 1: structure of bayes net
model = DiscreteBayesianNetwork([
    ('intelligence', 'grade'),
    ('studyhours', 'grade'),
    ('difficulty', 'grade'),
    ('grade', 'pass')
])

# Step 2: defining CPDs (conditional probability distribution)
cpd_intelligence = TabularCPD(variable='intelligence', variable_card = 2,
                             values=[[0.7], [0.3]], state_names={'intelligence': ['high', 'low']})

cpd_studyhours = TabularCPD(variable='studyhours', variable_card=2,
                           values=[[0.6],[0.4]], state_names={'studyhours':['sufficient', 'insufficient']})

cpd_difficulty = TabularCPD(variable='difficulty', variable_card=2,
                           values=[[0.4],[0.6]], state_names={'difficulty':['hard', 'easy']})

#P(grade | intelligence, studyhours, difficulty)
cpd_grade = TabularCPD(variable='grade', variable_card=3,
                      values=[
                          [0.1, 0.6, 0.2, 0.4, 0.2, 0.5, 0.1, 0.3],
                          [0.2, 0.2, 0.3, 0.5, 0.5, 0.4, 0.3, 0.4],
                          [0.7, 0.2, 0.5, 0.1, 0.3, 0.1, 0.6, 0.3]
                      ],
                      evidence=['intelligence', 'studyhours', 'difficulty'],
                      evidence_card=[2, 2, 2],
                      state_names={
                          'grade': ['A', 'B', 'C'],
                          'intelligence': ['high', 'low'],
                          'studyhours':['sufficient', 'insufficient'],
                          'difficulty':['hard', 'easy']
                      })

# P(pass | grade)
cpd_pass = TabularCPD(variable='pass', variable_card=2,
                     values=[
                         [0.05, 0.2, 0.5],
                         [0.95, 0.80, 0.50]
                     ],
                      evidence=['grade'],
                      evidence_card=[3],
                      state_names={
                          'pass': ['no', 'yes'], 'grade': ['A', 'B', 'C']
                      }
                     )

# Step 3: add cpds to the model
model.add_cpds(cpd_intelligence, cpd_studyhours, cpd_difficulty, cpd_grade, cpd_pass)

# Step 4: verify the model
assert model.check_model(), "Model is incorrect"

# Step5: perform inference
inference = VariableElimination(model)

# query: the probability that the student passes the exam, given: StudyHours = Sufficient, Difficulty = Hard
result = inference.query(variables=['pass'], evidence={'studyhours': 'sufficient', 'difficulty': 'hard'})
print(result)
print("\n")

# query: What is the probability that the student has High Intelligence, given: Pass = Yes
result2 = inference.query(variables={'intelligence':'high'}, evidence={'pass':'yes'})
print(result2)

+-----------+-------------+
| pass      |   phi(pass) |
+===========+=============+
| pass(no)  |      0.3545 |
+-----------+-------------+
| pass(yes) |      0.6455 |
+-----------+-------------+


+--------------------+---------------------+
| intelligence       |   phi(intelligence) |
+====================+=====================+
| intelligence(high) |              0.6965 |
+--------------------+---------------------+
| intelligence(low)  |              0.3035 |
+--------------------+---------------------+


## Task 3:

In [10]:
disease = ['Flu', 'Cold']
symptom = ['Yes', 'No']

model = DiscreteBayesianNetwork([
  ('Disease', 'Fever'),
  ('Disease', 'Cough'),
  ('Disease', 'Fatigue',),
  ('Disease', 'Chills')
])

disease_cpd = TabularCPD(variable='Disease', variable_card=2, values=[[0.3],[0.7]], state_names={'Disease': disease})

# P(Fever | Disease)
fever_cpd = TabularCPD(variable='Fever', variable_card=2, values=[[0.9, 0.5], [0.1, 0.5]], evidence=['Disease'], 
                       evidence_card=[2], state_names={'Fever': symptom, 'Disease': disease})

# P(Cough | Disease)
cough_cpd = TabularCPD(variable='Cough', variable_card=2, values=[[0.8, 0.6], [0.2, 0.4]], evidence=['Disease'],
                       evidence_card=[2], state_names={'Cough': symptom, 'Disease': disease})

# P(Chills | Disease)
chills_cpd = TabularCPD(variable='Chills', variable_card=2, values=[[0.6, 0.4], [0.4, 0.6]], evidence=['Disease'],
                        evidence_card=[2], state_names={'Chills': symptom, 'Disease': disease})

# P(Fatigue | Disease)
fatigue_cpd = TabularCPD(variable='Fatigue', variable_card=2, values=[[0.7, 0.3], [0.3, 0.7]], evidence=['Disease'],
                         evidence_card=[2], state_names={'Fatigue': symptom,'Disease': disease})

model.add_cpds(fever_cpd, disease_cpd, chills_cpd, cough_cpd, fatigue_cpd)

assert model.check_model(), "Model incorrect"

inference = VariableElimination(model)

result1 = inference.query(variables=['Disease'], evidence={'Fever': 'Yes', 'Cough': 'Yes'})
result2 = inference.query(variables=['Disease'], evidence={'Fever': 'Yes', 'Cough': 'Yes', 'Chills': 'Yes'})
result3 = inference.query(variables=['Fatigue'], evidence={'Disease': 'Flu'})

print(result1)
print("\n")
print(result2)
print("\n")
print(result3) 

+---------------+----------------+
| Disease       |   phi(Disease) |
+===============+================+
| Disease(Flu)  |         0.5070 |
+---------------+----------------+
| Disease(Cold) |         0.4930 |
+---------------+----------------+


+---------------+----------------+
| Disease       |   phi(Disease) |
+===============+================+
| Disease(Flu)  |         0.6067 |
+---------------+----------------+
| Disease(Cold) |         0.3933 |
+---------------+----------------+


+--------------+----------------+
| Fatigue      |   phi(Fatigue) |
+==============+================+
| Fatigue(Yes) |         0.7000 |
+--------------+----------------+
| Fatigue(No)  |         0.3000 |
+--------------+----------------+


## Task 4:

In [13]:
import random

# Define weather states
states = ["Sunny", "Cloudy", "Rainy"]

# Transition probabilities: P(next_state | current_state)
transition_matrix = {
    "Sunny": {"Sunny": 0.6, "Cloudy": 0.3, "Rainy": 0.1},
    "Cloudy": {"Sunny": 0.2, "Cloudy": 0.5, "Rainy": 0.3},
    "Rainy": {"Sunny": 0.1, "Cloudy": 0.4, "Rainy": 0.5}
}

def next_weather(current_state):
    transitions = transition_matrix[current_state]
    weather, probs = zip(*transitions.items())
    return random.choices(weather, weights=probs)[0]

def simulate_weather(start_state, days=10):
    sequence = [start_state]
    current = start_state
    for _ in range(days - 1):
        current = next_weather(current)
        sequence.append(current)
    return sequence

def estimate_rainy_days_probability(required_rainy=3, days=10, trials=1000):
    count = 0
    for _ in range(trials):
        forecast = simulate_weather("Sunny", days)
        if forecast.count("Rainy") >= required_rainy:
            count += 1
    return count / trials

if __name__ == "__main__":
    print("=== Simulated Weather for 10 Days (Starting from Sunny) ===")
    weather_sequence = simulate_weather("Sunny", 10)
    print(" → ".join(weather_sequence))

    print("\n=== Estimating Probability of at Least 3 Rainy Days ===")
    probability = estimate_rainy_days_probability()
    print(f"Estimated probability (based on 1000 runs): {probability:.4f}")


=== Simulated Weather for 10 Days (Starting from Sunny) ===
Sunny → Sunny → Cloudy → Sunny → Sunny → Sunny → Sunny → Cloudy → Cloudy → Cloudy

=== Estimating Probability of at Least 3 Rainy Days ===
Estimated probability (based on 1000 runs): 0.3980
